# N2 · 可复现性审计 + 确定性验证 (Reproducibility Audit)

> 配套 9.5-L1/L4 · **真实科研动作**: 用 `repro_check` 给规范/潦草两条记录做可复现体检;
> 亲手验证「同 config+seed 两次一致 / 不固定 seed 则不一致」; 再拿 checklist 审视自己的复现专题。

In [1]:
import sys
from pathlib import Path
SRC = Path.cwd().parent / "src"
EXP_SRC = Path.cwd().parent.parent / "experiment-design" / "src"   # 复用 9.4 的模拟器
for p in (SRC, EXP_SRC):
    sys.path.insert(0, str(p))
import numpy as np
import experiment as ex
import exp_tracker as et
import repro_check as rc
print("工具就绪")

工具就绪


## 1. 确定性验证: seed 是 Repeatable 的钥匙 (L1)

第一层可复现 (Repeatable) = 同代码同 config 同 seed, 重跑得到**完全一样**的数。
9.4 的模拟器是确定性的 —— 我们来证明它。

In [2]:
a = ex.run_experiment("Robust-DPO", 0.4, seed=3)
b = ex.run_experiment("Robust-DPO", 0.4, seed=3)
print(f"同 (method,noise,seed) 两次: {a:.6f}  vs  {b:.6f}  →  {'一致 ✅ (Repeatable)' if a==b else '不一致 ❌'}")

# 反面: 如果用"不固定的随机", 每次都不同 (这是没 seed everything 的下场)
rng_free = np.random.default_rng()   # 不给 seed
x, y = rng_free.normal(), rng_free.normal()
print(f"不固定 seed 的随机两次: {x:.4f}  vs  {y:.4f}  →  不一致 (连 Repeatable 都过不了)")

同 (method,noise,seed) 两次: 0.563006  vs  0.563006  →  一致 ✅ (Repeatable)
不固定 seed 的随机两次: -0.4171  vs  0.1097  →  不一致 (连 Repeatable 都过不了)


## 2. 回顾 9.4 踩过的真坑: hash(str) 的进程级随机盐 (L1)

9.4 的 `experiment.py` 一开始用 `hash((method, noise))` 派生种子, 结果**每次重跑 notebook 数字都变** ——
因为 Python 对字符串的内置 `hash()` 默认带进程级随机盐 (PYTHONHASHSEED)。这是个教科书级可复现 bug。
下面演示这个盐的存在 (注意: 同一进程内 hash 一致, 跨进程才变, 所以这里看的是"它确实参与了随机性")。

In [3]:
import os
print("PYTHONHASHSEED =", os.environ.get("PYTHONHASHSEED", "(未设置 → 默认随机盐)"))
print("→ 教训: 任何进入随机种子的东西, 自己必须是确定性的。")
print("→ 9.4 的修复: 用手工整数编码 (_stable_seed) 取代 hash(str)。seed_everything 也要设 PYTHONHASHSEED。")

PYTHONHASHSEED = (未设置 → 默认随机盐)
→ 教训: 任何进入随机种子的东西, 自己必须是确定性的。
→ 9.4 的修复: 用手工整数编码 (_stable_seed) 取代 hash(str)。seed_everything 也要设 PYTHONHASHSEED。


## 3. 可复现性体检: 规范 vs 潦草 (L4, repro_check)

In [4]:
# 取 N1 留痕的一条规范记录 (若 N1 没跑, 用一条等价的手造记录)
good = {"config": {"method": "Robust-DPO", "noise": 0.4, "seed": 3, "dataset": "sim-prefs@v1"},
        "git_sha": "a1b2c3d", "env": {"python": "3.13"}, "metrics": {"win_rate": 0.53}}
bad  = {"config": {"method": "Robust-DPO"}, "git_sha": "no-git", "metrics": {}}

print("=== 规范记录 ===")
print(rc.render(rc.audit(good)))
print("\n=== 潦草记录 ===")
print(rc.render(rc.audit(bad)))

=== 规范记录 ===
可复现性体检: 6/6

  ✅ 种子已固定
  ✅ 代码版本已记 (git sha)
  ✅ 超参配置完整
  ✅ 环境指纹已记
  ✅ 数据版本已记
  ✅ 结果已结构化记录

裁决: 可复现性良好

=== 潦草记录 ===
可复现性体检: 0/6

  ❌ 种子已固定
        → config 里没有 seed —— 加上并在代码里 seed everything
  ❌ 代码版本已记 (git sha)
        → 没记 git sha —— 用 exp_tracker.init 自动盖章
  ❌ 超参配置完整
        → config 为空/过简 —— 把所有超参显式写进 config
  ❌ 环境指纹已记
        → 没记 python/包版本 —— 记录 env 或附 lockfile
  ❌ 数据版本已记
        → 没记数据集版本/哈希 —— 记下 data 名称+版本+哈希
  ❌ 结果已结构化记录
        → metrics 为空 —— 别只截图, 把指标写进记录

裁决: 复现风险高: 半年后很可能重现不出来


## 4. 把 checklist 用在「真实的自己」身上

`repro_check` 也能审任意 dict。下面模拟「审视一个早期不规范的实验」: 只记了结果, 啥都没留。
体检告诉你**具体缺哪几项、怎么补** —— 这就是把模糊的「要可复现」变成可操作的清单。

In [5]:
my_early_run = {
    "config": {"method": "Robust-DPO", "lr": 1e-4},   # 漏了 seed 和 data 版本!
    "git_sha": "f00ba4-dirty",                         # 代码没提交
    "metrics": {"win_rate": 0.55},
}
report = rc.audit(my_early_run)
print(rc.render(report))
print(f"\n体检结论: {report['score']}/{report['total']} —— 主要缺: seed / 数据版本; git 还是 dirty。")
print("整改: ① config 加 seed+dataset@version ② 跑正式结果前 commit 代码。")

可复现性体检: 3/6

  ❌ 种子已固定
        → config 里没有 seed —— 加上并在代码里 seed everything
  ✅ 代码版本已记 (git sha) ⚠ 代码有未提交改动 (-dirty), 复现性打折
  ✅ 超参配置完整
  ❌ 环境指纹已记
        → 没记 python/包版本 —— 记录 env 或附 lockfile
  ❌ 数据版本已记
        → 没记数据集版本/哈希 —— 记下 data 名称+版本+哈希
  ✅ 结果已结构化记录

裁决: 复现风险高: 半年后很可能重现不出来

体检结论: 3/6 —— 主要缺: seed / 数据版本; git 还是 dirty。
整改: ① config 加 seed+dataset@version ② 跑正式结果前 commit 代码。


## 5. 反思 (9.5 收口)

你现在能: 区分三层可复现、写 seed_everything、用 config as code、用追踪器留痕、用 checklist 审计。
**复现纪律不是投稿前补的, 是从第一个实验就养的。**

**带走的最实在一条**: 把 `repro_check` (或 checklist 模板) 当成你每个进论文结果的 gate。
现在就拿它去审你某个 `learning/` 复现专题, 看缺哪几项 —— 那是你真实的、立刻能补的功课。

> 交棒 9.6: 结果设计对了(9.4)、可靠跑出并留痕了(9.5), 接下来把它**讲给人看** ——
> 下一专题 `research-figures` 教你把 9.4 那张交互效应图升级到能进顶会论文的出版级水准。